# <font color="#418FDE" size="6.5" uppercase>**Factor Components**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply dictionary learning, sparse coding, factor analysis, ICA, and NMF methods. 
- Use mini-batch and topic-modeling decompositions for larger or text-like data. 
- Interpret components, scores, and reconstruction quality. 


## **1. Dictionary Methods**

### **1.1. Dictionary Learning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_B/image_01_01.jpg?v=1784026418" width="250">



>* Learns reusable atoms directly from data
>* Reconstructs samples with compact meaningful components

>* Sparse atoms make representations clearer and robust
>* Useful for denoising, compression, and detection

>* Interpret atoms and coefficients together
>* Balance reconstruction, sparsity, and component usefulness



In [ ]:
#@title Python Code - Dictionary Learning

# Learn dictionary atoms from simple synthetic signals.
# Sparse codes use only a few atoms.
# Reconstruction error shows how well atoms combine.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import DictionaryLearning
import sklearn

# Create repeatable signals from three hidden building blocks.
rng = np.random.default_rng(42)
x_axis = np.linspace(0, 1, 40)

atom_a = np.sin(2 * np.pi * x_axis)
atom_b = np.where((x_axis > 0.25) & (x_axis < 0.55), 1.0, 0.0)
atom_c = np.exp(-80 * (x_axis - 0.75) ** 2)

true_atoms = np.vstack([atom_a, atom_b, atom_c])
true_atoms = true_atoms / np.linalg.norm(true_atoms, axis=1, keepdims=True)

# Each sample uses only two hidden atoms plus small noise.
coefficients = rng.uniform(0.0, 1.5, size=(90, 3))
mask = rng.random(size=(90, 3)) < 0.45
coefficients = coefficients * mask

signals = coefficients @ true_atoms
signals = signals + rng.normal(0.0, 0.04, size=signals.shape)

# Validate the small matrix before fitting the model.
if signals.shape != (90, 40):
    raise ValueError("Expected 90 signals with 40 measurements each.")

# Dictionary learning discovers reusable atoms and sparse codes.
model = DictionaryLearning(
    n_components=3,
    alpha=0.8,
    max_iter=500,
    transform_algorithm="lasso_lars",
    random_state=42,
)

sparse_codes = model.fit_transform(signals)
learned_atoms = model.components_
reconstructed = sparse_codes @ learned_atoms

# Summarize sparsity and reconstruction quality for beginners.
mean_active_atoms = np.mean(np.abs(sparse_codes) > 0.05)
mean_active_atoms = mean_active_atoms * sparse_codes.shape[1]
reconstruction_error = np.mean((signals - reconstructed) ** 2)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Signals shape: {signals.shape[0]} samples by {signals.shape[1]} features")
print(f"Average active atoms per signal: {mean_active_atoms:.2f} of 3")
print(f"Mean squared reconstruction error: {reconstruction_error:.4f}")

# Plot the learned atoms as reusable signal patterns.
fig, ax = plt.subplots(figsize=(8, 4))

for atom_index in range(learned_atoms.shape[0]):
    ax.plot(x_axis, learned_atoms[atom_index], label=f"Atom {atom_index + 1}")

ax.set_title("Dictionary Learning: Learned Atoms")
ax.set_xlabel("Signal position")
ax.set_ylabel("Atom value")
ax.legend()
plt.show()



### **1.2. Mini Batch Dictionary**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_B/image_01_02.jpg?v=1784026416" width="250">



>* Learns reusable patterns from small data batches
>* Scales dictionary learning to massive datasets

>* Balances speed, sparsity, and reconstruction quality
>* Scales to evolving, meaningful data patterns

>* Inspect components, codes, and reconstruction quality
>* Check batch stability and sparsity balance



In [ ]:
#@title Python Code - Mini Batch Dictionary

# Demonstrate mini batch dictionary learning on small images.
# Learn reusable atoms from many digit patches.
# Compare sparse reconstruction with the original data.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_digits
from sklearn.decomposition import MiniBatchDictionaryLearning
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

# Load packaged digit images for a fast offline example.
digits = load_digits()
images = digits.images[:300]

# Validate the expected image shape before modeling.
if images.shape[1:] != (8, 8):
    raise ValueError("Expected 8 by 8 digit images.")

# Flatten images and scale pixels to zero through one.
flat_images = images.reshape(images.shape[0], -1)
scaler = MinMaxScaler()
scaled_images = scaler.fit_transform(flat_images)

# Fit a compact dictionary using small mini batches.
dictionary_model = MiniBatchDictionaryLearning(
    n_components=16,
    alpha=1.0,
    batch_size=32,
    max_iter=80,
    random_state=42,
    transform_algorithm="lasso_lars",
    verbose=False,
)

# Sparse codes say which learned atoms each image uses.
sparse_codes = dictionary_model.fit_transform(scaled_images)
reconstructed = sparse_codes @ dictionary_model.components_

# Measure reconstruction quality and sparsity.
rmse = mean_squared_error(scaled_images, reconstructed) ** 0.5
active_counts = np.count_nonzero(np.abs(sparse_codes) > 0.001, axis=1)
average_active = active_counts.mean()

print(f"scikit-learn version: {sklearn_version}")
print(f"Dictionary atoms learned: {dictionary_model.components_.shape[0]}")
print(f"Average active atoms per image: {average_active:.2f}")
print(f"Reconstruction RMSE on scaled pixels: {rmse:.3f}")

# Show the learned atoms as tiny reusable image patterns.
atom_grid = np.zeros((32, 32))
for atom_index in range(16):
    row = atom_index // 4
    col = atom_index % 4
    atom = dictionary_model.components_[atom_index].reshape(8, 8)
    atom_grid[row * 8:(row + 1) * 8, col * 8:(col + 1) * 8] = atom

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(atom_grid, cmap="gray")
ax.set_title("Mini-batch dictionary atoms learned from digits")
ax.set_xlabel("Atom grid columns")
ax.set_ylabel("Atom grid rows")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



### **1.3. Sparse Coding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_B/image_01_03.jpg?v=1784026415" width="250">



>* Represent data with few dictionary patterns
>* Produce compact scores for each observation

>* Few components reveal key patterns
>* Useful for images, signals, and behavior

>* Balance detail preservation with sparse simplicity
>* Use components for interpretable structured representations



In [ ]:
#@title Python Code - Sparse Coding

# This example demonstrates sparse coding with learned components.
# Sparse codes use only a few active dictionary atoms.
# Reconstruction quality is compared with code sparsity.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.decomposition import DictionaryLearning
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

# Load small handwritten digit images packaged with scikit-learn.
digits = load_digits()
images = digits.images[:300]

# Flatten each image into one row of pixel features.
flat_images = images.reshape(images.shape[0], -1)

# Scale pixels so reconstruction errors are easy to compare.
scaler = MinMaxScaler()
scaled_images = scaler.fit_transform(flat_images)

# Validate the expected two-dimensional learning table.
if scaled_images.shape != (300, 64):
    raise ValueError("Expected 300 digit images with 64 pixel features.")

# Learn a dictionary and sparse code for every image.
model = DictionaryLearning(
    n_components=16,
    transform_algorithm="lasso_lars",
    transform_alpha=0.15,
    random_state=42,
    max_iter=80,
)

sparse_codes = model.fit_transform(scaled_images)
reconstructed = sparse_codes @ model.components_

# Count how many dictionary atoms are active per image.
active_counts = np.count_nonzero(np.abs(sparse_codes) > 0.001, axis=1)
mean_active = active_counts.mean()
error = mean_squared_error(scaled_images, reconstructed)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Dictionary atoms learned: {model.components_.shape[0]}")
print(f"Average active atoms per image: {mean_active:.2f} of 16")
print(f"Mean squared reconstruction error: {error:.4f}")

# Plot one sparse code to show most coefficients are zero.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(np.arange(16), sparse_codes[0])
ax.set_title("Sparse code for one digit image")
ax.set_xlabel("Dictionary atom index")
ax.set_ylabel("Coefficient strength")
plt.show()



## **2. Scalable Factor Models**

### **2.1. Latent Factor Analysis**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_B/image_02_01.jpg?v=1784026404" width="250">



>* Hidden factors explain related observed measurements
>* Reveals structure in noisy high-dimensional data

>* Update hidden factors using smaller data batches
>* Reveal document themes in massive text collections

>* Interpret factors through variables and scores
>* Balance useful meaning with reconstruction quality



In [ ]:
#@title Python Code - Latent Factor Analysis

# This example reveals hidden factors in synthetic data.
# Factor analysis estimates shared latent influences from measurements.
# The output compares true and learned factor patterns.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler

# A fixed generator makes the simulated lesson repeatable.
rng = np.random.default_rng(42)

# Three hidden factors influence eight observed measurements.
student_count = 600
factor_count = 3
feature_names = np.array([
    "attendance", "assignments", "forum", "quiz", "exam",
    "projects", "late_work", "help_requests"
])

# These loadings define the hidden structure we hope to recover.
true_loadings = np.array([
    [1.2, 0.1, -0.2],
    [1.1, 0.2, -0.1],
    [0.9, 0.0, 0.3],
    [0.1, 1.2, -0.1],
    [0.0, 1.3, -0.2],
    [0.2, 1.0, 0.1],
    [-0.2, -0.1, 1.2],
    [0.1, 0.0, 1.1]
])

# Observed data combine hidden scores with small measurement noise.
hidden_scores = rng.normal(size=(student_count, factor_count))
noise = rng.normal(scale=0.35, size=(student_count, len(feature_names)))
observed_data = hidden_scores @ true_loadings.T + noise

# Validate the simulated table before modeling.
if observed_data.shape != (student_count, len(feature_names)):
    raise ValueError("The simulated data table has an unexpected shape.")

# Scaling keeps variables comparable before factor analysis.
scaler = StandardScaler()
scaled_data = scaler.fit_transform(observed_data)

# FactorAnalysis learns latent factors from the observed measurements.
model = FactorAnalysis(n_components=factor_count, random_state=42)
learned_scores = model.fit_transform(scaled_data)
learned_loadings = model.components_.T

# Reconstruction quality shows how much structure the factors preserve.
reconstructed = learned_scores @ model.components_ + model.mean_
mean_squared_error = np.mean((scaled_data - reconstructed) ** 2)

# Print concise teaching results for interpretation.
print("scikit-learn version:", sklearn.__version__)
print("Observed measurements:", len(feature_names))
print("Latent factors learned:", factor_count)
print("Reconstruction MSE:", round(float(mean_squared_error), 3))

# Show which measurements define each learned factor most strongly.
for factor_index in range(factor_count):
    strengths = np.abs(learned_loadings[:, factor_index])
    top_indices = np.argsort(strengths)[-3:][::-1]
    top_names = ", ".join(feature_names[top_indices])
    print("Factor", factor_index + 1, "top measurements:", top_names)

# A heatmap makes factor interpretation visible at a glance.
fig, ax = plt.subplots(figsize=(8, 4))
image = ax.imshow(learned_loadings, cmap="coolwarm", aspect="auto")
ax.set_title("Learned latent factor loadings")
ax.set_xlabel("Latent factor")
ax.set_ylabel("Observed measurement")
ax.set_xticks(np.arange(factor_count))
ax.set_xticklabels(["F1", "F2", "F3"])
ax.set_yticks(np.arange(len(feature_names)))
ax.set_yticklabels(feature_names)
fig.colorbar(image, ax=ax, label="Loading strength")
plt.tight_layout()
plt.show()



### **2.2. Independent Component Analysis**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_B/image_02_02.jpg?v=1784026406" width="250">



>* ICA separates mixed data into independent sources
>* Useful for audio, signals, finance, and text

>* Scale ICA with whitening and chunked processing
>* Preprocess text carefully; compare topic alternatives

>* Components reveal latent sources across variables
>* Balance reconstruction, scalability, stability, and interpretability



In [ ]:
#@title Python Code - Independent Component Analysis

# This example separates mixed signals using ICA.
# Independent components recover hidden source patterns.
# The plot compares mixtures with recovered sources.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.decomposition import FastICA
from sklearn.preprocessing import StandardScaler

# Create three independent source signals over time.
rng = np.random.default_rng(42)
time = np.linspace(0, 8, 800)
source_one = np.sin(2 * np.pi * time)
source_two = np.sign(np.sin(3 * np.pi * time))
source_three = rng.laplace(size=time.size)

# Stack sources as rows for easier mixing.
true_sources = np.vstack([source_one, source_two, source_three])
mixing_matrix = np.array([[1.0, 0.5, 0.2], [0.4, 1.0, 0.5], [0.2, 0.3, 1.0]])
mixed_signals = mixing_matrix @ true_sources

# Validate the small teaching dataset shape.
if mixed_signals.shape != (3, time.size):
    raise ValueError("Expected three mixed signals across all time points.")

# Scale each observed mixture before fitting ICA.
scaler = StandardScaler()
observations = scaler.fit_transform(mixed_signals.T)

# Fit ICA to estimate statistically independent sources.
ica = FastICA(n_components=3, random_state=42, max_iter=500, whiten="unit-variance")
estimated_sources = ica.fit_transform(observations)
reconstructed = ica.inverse_transform(estimated_sources)

# Measure reconstruction error in the scaled observation space.
error = np.mean((observations - reconstructed) ** 2)
print("scikit-learn version:", sklearn.__version__)
print("Scaled reconstruction MSE:", round(float(error), 4))
print("ICA output shape:", estimated_sources.shape)

# Plot one mixture and one recovered independent component.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(time, observations[:, 0], label="observed mixture", alpha=0.75)
ax.plot(time, estimated_sources[:, 0], label="recovered component", alpha=0.75)
ax.set_title("ICA separates a mixed signal into independent components")
ax.set_xlabel("Time")
ax.set_ylabel("Scaled signal value")
ax.legend()
plt.show()



### **2.3. Nonnegative Matrix Factorization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_B/image_02_03.jpg?v=1784026408" width="250">



>* Models nonnegative data as additive components
>* Reveals interpretable themes in text-like data

>* Compress large data into interpretable components
>* Reveal topics, visual parts, and preferences

>* Balance interpretability with reconstruction accuracy
>* Use mini-batches for scalable NMF



In [ ]:
#@title Python Code - Nonnegative Matrix Factorization

# This example builds topics with nonnegative matrix factorization.
# NMF finds additive components in count data.
# The output shows topic words and reconstruction error.

import numpy as np
import sklearn
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import mean_squared_error

# These short documents form a tiny text collection.
documents = [
    "stocks market profit bank finance investment",
    "bank loan market finance profit stocks",
    "election vote government policy senate campaign",
    "campaign election vote policy government debate",
    "team score game coach player season",
    "game team player score tournament coach",
]

# CountVectorizer creates a nonnegative document term matrix.
vectorizer = CountVectorizer()
term_matrix = vectorizer.fit_transform(documents)
feature_names = np.array(vectorizer.get_feature_names_out())

# NMF summarizes documents as mixtures of three nonnegative topics.
model = NMF(n_components=3, init="nndsvda", random_state=42, max_iter=500)
document_topics = model.fit_transform(term_matrix)
topic_words = model.components_

# Reconstruct the original counts from document scores and topics.
reconstructed = np.dot(document_topics, topic_words)
error = mean_squared_error(term_matrix.toarray(), reconstructed)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Document-term matrix shape: {term_matrix.shape}")
print(f"Mean squared reconstruction error: {error:.3f}")

# The largest weights reveal the most representative words per topic.
for topic_index, weights in enumerate(topic_words, start=1):
    top_indices = np.argsort(weights)[-4:][::-1]
    words = ", ".join(feature_names[top_indices])
    print(f"Topic {topic_index}: {words}")

# Each document receives nonnegative topic scores.
first_scores = np.round(document_topics[0], 2)
print(f"First document topic scores: {first_scores}")



## **3. Topic Reconstruction**

### **3.1. Mini Batch Reconstruction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_B/image_03_01.jpg?v=1784026411" width="250">



>* Learn components from small data batches
>* Rebuild data while preserving key structure

>* Components build reconstructions; scores show contribution strength
>* Quality varies with patterns, rarity, and training

>* Judge error alongside component meaning
>* Balance efficiency, accuracy, and interpretability



In [ ]:
#@title Python Code - Mini Batch Reconstruction

# Mini batches learn reusable reconstruction components.
# Scores combine components to approximate observations.
# Reconstruction error shows preserved data structure.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import MiniBatchDictionaryLearning
from sklearn.metrics import mean_squared_error
import sklearn

# Small synthetic documents keep the example fast and clear.
rng = np.random.default_rng(42)
topic_words = np.array([
    [8, 7, 6, 1, 1, 1],
    [1, 1, 1, 7, 8, 6],
    [5, 1, 4, 5, 1, 4],
])

# Each row is a document built from hidden topic weights.
true_scores = rng.gamma(shape=1.5, scale=1.0, size=(60, 3))
word_counts = true_scores @ topic_words
word_counts = word_counts + rng.normal(0, 0.6, size=word_counts.shape)
word_counts = np.clip(word_counts, 0, None)

# Validate the matrix before fitting the decomposition model.
if word_counts.shape != (60, 6):
    raise ValueError("Expected 60 documents and 6 word features.")

# MiniBatchDictionaryLearning updates components using small batches.
model = MiniBatchDictionaryLearning(
    n_components=3,
    batch_size=10,
    max_iter=200,
    random_state=42,
)

# Scores describe how strongly each component appears in each document.
learned_scores = model.fit_transform(word_counts)
reconstructed_counts = learned_scores @ model.components_

# Reconstruction error summarizes how much information was lost.
mse = mean_squared_error(word_counts, reconstructed_counts)
relative_error = np.linalg.norm(word_counts - reconstructed_counts)
relative_error = relative_error / np.linalg.norm(word_counts)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Documents x word features: {word_counts.shape[0]} x {word_counts.shape[1]}")
print(f"Mean squared reconstruction error: {mse:.3f}")
print(f"Relative reconstruction error: {relative_error:.3f}")
print("First document component scores:", np.round(learned_scores[0], 2))

# Compare one original document with its mini-batch reconstruction.
word_labels = ["word 1", "word 2", "word 3", "word 4", "word 5", "word 6"]
x_positions = np.arange(len(word_labels))
bar_width = 0.38

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x_positions - bar_width / 2, word_counts[0], bar_width, label="Original")
ax.bar(x_positions + bar_width / 2, reconstructed_counts[0], bar_width, label="Reconstructed")
ax.set_title("Mini-Batch Reconstruction of One Synthetic Document")
ax.set_xlabel("Word feature")
ax.set_ylabel("Approximate count")
ax.set_xticks(x_positions)
ax.set_xticklabels(word_labels)
ax.legend()
plt.tight_layout()
plt.show()



### **3.2. LDA Topic Reconstruction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_B/image_03_02.jpg?v=1784026409" width="250">



>* Documents become mixtures of shared topics
>* Topics summarize themes and reconstruction strength

>* Inspect topic words to identify themes
>* Use document scores to judge topic fit

>* Balance topic count for coherent reconstructions
>* Prefer stable, useful, interpretable topic summaries



In [ ]:
#@title Python Code - LDA Topic Reconstruction

# This example reconstructs documents with LDA topics.
# Topic mixtures explain word-count patterns compactly.
# Printed results compare original and reconstructed counts.

import numpy as np
import sklearn
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# Small documents make the topic reconstruction easy to inspect.
documents = [
    "team scored goal match coach league",
    "player goal team win match season",
    "election vote policy government debate",
    "senate election policy vote campaign",
    "recipe kitchen flavor dinner ingredient",
    "bake recipe kitchen meal flavor",
]

# CountVectorizer turns text into a document-word count matrix.
vectorizer = CountVectorizer()
word_counts = vectorizer.fit_transform(documents)

# Validate the matrix before fitting the decomposition model.
if word_counts.shape[0] != len(documents):
    raise ValueError("Each document should become one matrix row.")

# LDA learns topics and each document's topic mixture.
lda = LatentDirichletAllocation(
    n_components=3,
    random_state=42,
    learning_method="batch",
    max_iter=30,
)

document_topics = lda.fit_transform(word_counts)

# Multiplying mixtures by topics gives expected reconstructed word counts.
topic_word_probabilities = lda.components_ / lda.components_.sum(axis=1)[:, None]
reconstructed_counts = document_topics @ topic_word_probabilities
reconstructed_counts = reconstructed_counts * word_counts.sum(axis=1).A

# Show the strongest words for each learned topic.
words = np.array(vectorizer.get_feature_names_out())
print("scikit-learn version:", sklearn.__version__)

for topic_index, topic in enumerate(topic_word_probabilities):
    top_word_indices = np.argsort(topic)[-4:][::-1]
    top_words = ", ".join(words[top_word_indices])
    print("Topic", topic_index + 1, "top words:", top_words)

# Compare one document's observed counts with its LDA reconstruction.
document_index = 0
important_indices = np.argsort(word_counts[document_index].toarray()[0])[-5:][::-1]

print("Document 1 topic mixture:", np.round(document_topics[document_index], 2))
print("Word | observed | reconstructed")

for word_index in important_indices:
    observed = int(word_counts[document_index, word_index])
    reconstructed = round(float(reconstructed_counts[document_index, word_index]), 2)
    print(words[word_index], "|", observed, "|", reconstructed)

# A smaller error means the topic mixture explains counts better.
error = np.linalg.norm(word_counts.toarray() - reconstructed_counts)
print("Overall reconstruction error:", round(float(error), 2))



### **3.3. Interpreting Components**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_B/image_03_03.jpg?v=1784026413" width="250">



>* Name components from their strongest features
>* Treat interpretations as domain-informed judgments

>* Scores show each observation’s topic strength
>* Interpret scores using context and distribution

>* Balance reconstruction accuracy with interpretability
>* Check components, scores, and usefulness together



In [ ]:
#@title Python Code - Interpreting Components

# This example interprets topic components from tiny documents.
# High weights reveal words defining each component.
# Scores show which documents use each topic.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import CountVectorizer

# These short documents contain two easy themes.
documents = [
    "vote election campaign candidate debate policy",
    "election vote policy government campaign",
    "candidate debate vote election",
    "battery screen camera device update",
    "screen camera battery device review",
    "device update battery screen",
]

# CountVectorizer turns words into a document-word matrix.
vectorizer = CountVectorizer()
word_counts = vectorizer.fit_transform(documents)
feature_names = np.array(vectorizer.get_feature_names_out())

# Validate the matrix before fitting the decomposition.
if word_counts.shape[0] != len(documents):
    raise ValueError("Each document needs one row.")

# NMF finds nonnegative topics and document scores.
model = NMF(n_components=2, init="nndsvda", random_state=42, max_iter=300)
document_scores = model.fit_transform(word_counts)
components = model.components_

# Reconstruction error summarizes numerical fit quality.
reconstructed = document_scores @ components
original = word_counts.toarray()
reconstruction_error = np.linalg.norm(original - reconstructed)

print("scikit-learn version:", sklearn.__version__)
print("Reconstruction error:", round(float(reconstruction_error), 2))

# Show the strongest words for each learned component.
for topic_index in range(2):
    top_indices = np.argsort(components[topic_index])[-5:][::-1]
    top_words = ", ".join(feature_names[top_indices])
    print("Topic", topic_index + 1, "top words:", top_words)

# Plot scores to connect documents with interpreted topics.
fig, ax = plt.subplots(figsize=(7, 4))
x_positions = np.arange(len(documents))
ax.bar(x_positions - 0.18, document_scores[:, 0], width=0.36, label="Topic 1")
ax.bar(x_positions + 0.18, document_scores[:, 1], width=0.36, label="Topic 2")

ax.set_title("Document Scores for Interpreted Components")
ax.set_xlabel("Document number")
ax.set_ylabel("Topic score")
ax.set_xticks(x_positions)
ax.set_xticklabels(["D1", "D2", "D3", "D4", "D5", "D6"])
ax.legend()

plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Factor Components**</font>


In this lecture, you learned to:
- Apply dictionary learning, sparse coding, factor analysis, ICA, and NMF methods. 
- Use mini-batch and topic-modeling decompositions for larger or text-like data. 
- Interpret components, scores, and reconstruction quality. 

In the next Module (Module 25), we will go over 'Manifold Learning'